# Perceptron

In [1]:
import numpy as np
from sklearn.linear_model import Perceptron

## 1.生成数据

In [2]:
w_true = np.array([1.5, -2.0, 0.8, 1.2, -1.0])
b_true = 0.3
n_samples = 80
margin = 0.5
rng = np.random.default_rng(seed=42)
n_pos = n_samples // 2
n_neg = n_samples - n_pos
X_pos = []
X_neg = []
while len(X_pos) < n_pos or len(X_neg) < n_neg:
    x = rng.normal(loc = 0.0, scale = 1.0, size = 5)
    score = np.dot(w_true, x) + b_true

    if abs(score) < margin:
        continue

    y = 1 if score >= 0 else -1

    if y == 1 and len(X_pos) < n_pos:
        X_pos.append(x)
    elif y == -1 and len(X_neg) < n_neg:
        X_neg.append(x)

X_train = np.vstack([X_pos, X_neg])
y_train = np.array([[1] * n_pos + [-1] * n_neg])

print("真实参数：")
print("w_true =", w_true)
print("b_true =", b_true)

print("\n训练数据形状：")
print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)

print("\n类别数量：")
print("正类数量 =", np.sum(y_train == 1))
print("负类数量 =", np.sum(y_train == -1))

真实参数：
w_true = [ 1.5 -2.   0.8  1.2 -1. ]
b_true = 0.3

训练数据形状：
X_train.shape = (80, 5)
y_train.shape = (1, 80)

类别数量：
正类数量 = 40
负类数量 = 40


## 感知机模型

In [3]:
class MyPerceptron:
    def __init__(self, eta = 0.1, max_epochs = 100, shuffle = True,
                 random_state = 0, verbose = True, max_print_updates = 1000):
        self.eta = eta # learning rate
        self.max_epochs = max_epochs # 最大迭代轮数，一轮epoch的意思是：把所有训练样本都看一遍
        self.shuffle = shuffle # 每一轮训练之前都打乱样本顺序
        self.random_state = random_state # 如果设置相同的 random_state，每次运行代码时打乱顺序的结果相同
        self.verbose = verbose # verbose=True 表示训练时打印详细过程
        self.max_print_updates = max_print_updates # 控制最多打印多少次更新过程
        self.w_ = None
        self.b_ = None
        self.n_epochs_ = 0 # 最终训练了多少轮
        self.n_updates_ = 0 # 一共发生了多少次误分类更新
        self.history_ = [] # 保存训练过程中的每一次更新记录

    # fit 函数：核心训练过程
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # 初始化
        self.w_ = np.zeros(n_features)
        self.b_ = 0.0
        rng = np.random.default_rng(self.random_state) # 用于之后打乱样本顺序
        total_update = 0 # 记录总共更新了多少次
        # 批量进行训练
        for epoch in range(1, self.max_epochs + 1):
            errors = 0 # 用来记录当前这一轮发生了多少次误分类更新

            indices = np.arange(n_samples) # n_samples的索引
            if self.shuffle:
                rng.shuffle(indices) # 打乱索引，这是一个原地打乱顺序的方法

            if self.verbose:
                print('=' * 80)
                print(f'Epoch {epoch}')
                print('=' * 80)

            for i in indices:
                x_i = X[i]
                y_i = y[i]

                score = np.dot(self.w_, x_i) + self.b_
                margin_value = y_i * score
                if margin_value <= 0:
                    old_w = self.w_.copy() # 为了打印更新前后的变化，用copy是保证old_w不会跟着变
                    old_b = self.b_

                    self.w_ = self.w_ + self.eta * y_i * x_i # 更新参数
                    self.b_ = self.b_ + self.eta * y_i

                    errors += 1 # 当前 epoch 内更新了多少次
                    total_update += 1 # 整个训练过程一共更新了多少次

                    loss = self.perceptron_loss(X, y) # 计算当前的训练损失
                    acc = self.score(X, y) # 计算当前的训练准确率

                    self.history_.append({'update': total_update,
                                          'epoch': epoch,
                                          'sample_index': i,
                                          'y_i':y_i,
                                          'old_margin': margin_value,
                                          'w':self.w_.copy(),
                                          'b': self.b_,
                                          'loss': loss,
                                          'accuracy': acc}) # 保存当前更新的详细信息到 history_ 中
                    if self.verbose and total_update <= self.max_print_updates:
                        print(f"第 {total_update:03d} 次更新")
                        print(f"样本编号 i = {i}")
                        print(f"真实标签 y_i = {y_i:+d}")
                        print(f"更新前 y_i(w·x_i+b) = {margin_value:.6f} <= 0，发生误分类")
                        print(f"更新前 w = {np.round(old_w, 4)}, b = {old_b:.4f}")
                        print(f"更新后 w = {np.round(self.w_, 4)}, b = {self.b_:.4f}")
                        print(f"当前训练损失 loss = {loss:.6f}")
                        print(f"当前训练准确率 accuracy = {acc:.4f}")
                        print("-" * 80)
            epoch_acc = self.score(X, y) # 每一轮结束后，重新计算当前训练集上的准确率和损失。
            epoch_loss = self.perceptron_loss(X, y)

            if self.verbose:
                print(f"Epoch {epoch} 结束")
                print(f"本轮误分类更新次数 = {errors}")
                print(f"当前 loss = {epoch_loss:.6f}")
                print(f"当前 accuracy = {epoch_acc:.4f}")
                print()

            if errors == 0: # 收敛判断
                if self.verbose:
                    print("训练结束：本轮没有误分类点，感知机已经收敛")
                self.n_epochs_ = epoch # # 记录训练轮数和总更新次数
                self.n_updates_ = total_update
                return self

        self.n_epochs_ = self.max_epochs # 记录训练轮数和总更新次数
        self.n_updates_ = total_update

        if self.verbose:
            print("达到最大迭代轮数，训练停止")
        return self

    def predict(self, X):
        scores = X @ self.w_ + self.b_
        return np.where(scores >= 0, 1, -1)

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

    def perceptron_loss(self, X, y):
        scores = X @ self.w_ + self.b_
        margins = y * scores

        misclassified = margins <= 0 # 这是布尔值

        if np.sum(misclassified) == 0:
            return 0.0

        loss = -np.sum(margins[misclassified])

        return loss

## 训练结果比较

In [ ]:
my_model = MyPerceptron(eta = 0.1, max_epochs = 100, shuffle = True,
                        random_state = 0, verbose = False, max_print_updates = 100)
y_train = np.asarray(y_train).ravel() # 在训练前把 y_train 拉平成一维
my_model.fit(X_train, y_train)
print()
print("=" * 80)
print("手写感知机训练结果")
print("=" * 80)
print(f"训练轮数 n_epochs = {my_model.n_epochs_}")
print(f"总更新次数 n_updates = {my_model.n_updates_}")
print(f"训练得到的 w = {np.round(my_model.w_, 6)}")
print(f"训练得到的 b = {my_model.b_:.6f}")
print(f"训练集准确率 = {my_model.score(X_train, y_train):.4f}")
print(f"最终感知机损失 = {my_model.perceptron_loss(X_train, y_train):.6f}")
print()
theta_true = np.append(w_true, b_true)
theta_learned = np.append(my_model.w_, my_model.b_)
raw_diff = np.linalg.norm(theta_true - theta_learned)
norm_true = np.linalg.norm(theta_true)
norm_learned = np.linalg.norm(theta_learned)
if norm_true == 0 or norm_learned == 0:
    cosine_similarity = np.nan
    angle_degree = np.nan
else:
    cosine_similarity = np.dot(theta_true, theta_learned) / (norm_true * norm_learned)
    cosine_similarity = np.clip(cosine_similarity, -1.0, 1.0) # 把数据强行框定在一个指定的范围之内
    angle_degree = np.degrees(np.arccos(cosine_similarity)) # 计算夹角的度数

scale = np.dot(theta_true, theta_learned) / np.dot(theta_learned, theta_learned) # 计算最佳缩放比例 scale
theta_learned_scaled = scale * theta_learned # 把 theta_learned 按照 scale 进行缩放，得到 theta_learned_scaled
scaled_diff = np.linalg.norm(theta_true - theta_learned_scaled)


print(f"真实 w_true      = {np.round(w_true, 6)}")
print(f"真实 b_true      = {b_true:.6f}")
print()
print(f"训练得到 w       = {my_model.w_}")
print(f"训练得到 b       = {my_model.b_:.6f}")
print()
print(f"直接参数差距 ||theta_true - theta_learned|| = {raw_diff:.6f}")
print(f"方向余弦 cosine similarity = {cosine_similarity:.6f}")
print(f"夹角 angle = {angle_degree:.6f} 度")
print(f"缩放后的最小参数差距 = {scaled_diff:.6f}")
print()


手写感知机训练结果
训练轮数 n_epochs = 3
总更新次数 n_updates = 11
训练得到的 w = [ 0.237011 -0.489788  0.039765  0.210568 -0.226113]
训练得到的 b = 0.100000
训练集准确率 = 1.0000
最终感知机损失 = 0.000000

真实 w_true      = [ 1.5 -2.   0.8  1.2 -1. ]
真实 b_true      = 0.300000

训练得到 w       = [ 0.23701077 -0.48978833  0.03976451  0.21056799 -0.22611255]
训练得到 b       = 0.100000

直接参数差距 ||theta_true - theta_learned|| = 2.464085
方向余弦 cosine similarity = 0.962551
夹角 angle = 15.729741 度
缩放后的最小参数差距 = 0.832061

说明：
1. 感知机训练出来的参数不一定等于你规定的真实参数。
2. 因为只要超平面能把数据分开即可，解通常不是唯一的。
3. 所以更应该看训练准确率、方向余弦、夹角和缩放后的差距。



## 使用sklearn的Perceptron API训练模型

In [5]:
sklearn_model = Perceptron(penalty = None, fit_intercept = True, max_iter = 100, tol = None,
                           shuffle = True, eta0 = 0.1, random_state = 0, verbose = 0)
# penalty = None:不使用正则化项
# fit_intercept = True:训练时要学习偏置项 b
# max_iter = 100:最多训练 100 轮
# tol 是停止准则中的容忍度
# verbose = 1:表示 sklearn 训练时会打印一些训练过程信息
sklearn_model.fit(X_train, y_train)
sklearn_w = sklearn_model.coef_[0] # 取出 sklearn 训练得到的权重
sklearn_b = sklearn_model.intercept_[0] # 取出 sklearn 训练得到的偏置

print()
print("=" * 80)
print("sklearn 感知机训练结果")
print("=" * 80)
print(f"sklearn 训练轮数 n_iter_ = {sklearn_model.n_iter_}")
print(f"训练得到的 w = {np.round(sklearn_w, 6)}")
print(f"训练得到的 b = {sklearn_b:.6f}")
print(f"训练集准确率 = {sklearn_model.score(X_train, y_train):.4f}")
print()

my_pred = my_model.predict(X_train)
sklearn_pred = sklearn_model.predict(X_train)

print("=" * 80)
print("手写感知机 vs sklearn 感知机")
print("=" * 80)
print(f"两者在训练集上的预测是否完全一致：{np.all(my_pred == sklearn_pred)}")
print(f"两者预测相同的比例：{np.mean(my_pred == sklearn_pred):.4f}")
print()

print("前 10 个样本的预测结果：")
print("编号 | 真实标签 | 手写感知机预测 | sklearn预测")
print("-" * 80)
for i in range(10):
    print(f"{i:02d}   | {y_train[i]:+d}       | {my_pred[i]:+d}             | {sklearn_pred[i]:+d}")


sklearn 感知机训练结果
sklearn 训练轮数 n_iter_ = 100
训练得到的 w = [ 0.335179 -0.51041   0.158589  0.361766 -0.295316]
训练得到的 b = 0.000000
训练集准确率 = 1.0000

手写感知机 vs sklearn 感知机
两者在训练集上的预测是否完全一致：True
两者预测相同的比例：1.0000

前 10 个样本的预测结果：
编号 | 真实标签 | 手写感知机预测 | sklearn预测
--------------------------------------------------------------------------------
00   | +1       | +1             | +1
01   | +1       | +1             | +1
02   | +1       | +1             | +1
03   | +1       | +1             | +1
04   | +1       | +1             | +1
05   | +1       | +1             | +1
06   | +1       | +1             | +1
07   | +1       | +1             | +1
08   | +1       | +1             | +1
09   | +1       | +1             | +1
